In [1]:
import pandas as pd

In [2]:
# Charger le fichier Excel
df_macro_raw= pd.read_excel("data\macro_data\Données_macro_hist_v3.xlsx")

# Preprocessing du dataframe
df = df_macro_raw.copy()

id_cols = ["Region", "Variable", "Unit"]
value_cols = [c for c in df.columns if c not in id_cols]

df_long = df.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name="quarter",
    value_name="value"
)


In [4]:
df_long["Region"].unique()

array(['Europe', 'Japan', 'United States', 'World'], dtype=object)

Nous allons récupérer les variables macroéconomiques à considérer pour la modélisation des facteurs systémiques. D'abord, au regard des données dont nous disposons, nous ne pouvons considérer que les variables provenant des régions d'EUrope (5 variables disponibles) et des US (9 variables disponibles).

In [6]:
# Sélection des régions
regions_keep = ["United States", "Japan", "Europe"]
df = df_macro_raw[df_macro_raw["Region"].isin(regions_keep)].copy()
# Colonnes temporelles
time_cols = [c for c in df.columns if "-" in c]
# On garde uniquement les lignes sans NaN sur toute la période
df = df.dropna(subset=time_cols)
# Mapping régions → suffixes
region_map = {
    "United States": "US",
    "Japan": "JP",
    "Europe": "EU"
}

def clean_var_name(var):
    return (
        var.strip()
           .replace(" ", "_")
           .replace("(", "")
           .replace(")", "")
           .replace("/", "_")
    )

df["var_name"] = (
    df["Region"].map(region_map)
    + "_"
    + df["Variable"].apply(clean_var_name)
)


# Passage au format long
df_long = df.melt(
    id_vars=["var_name"],
    value_vars=time_cols,
    var_name="date",
    value_name="value"
)

# Pivot final
df_ts = df_long.pivot(
    index="date",
    columns="var_name",
    values="value"
).sort_index()

# Conversion "2010-Q1" → PeriodIndex trimestriel
df_ts.index = pd.PeriodIndex(df_ts.index, freq="Q").to_timestamp()

In [29]:
df_ts.columns.to_list()

['EU_Central_bank_Intervention_rate_policy_interest_rate',
 'EU_Effective_exchange_rate',
 'EU_GDP_Growth_Rate',
 'EU_Long_term_interest_rate',
 'EU_Unemployment_rate',
 'JP_Central_bank_Intervention_rate_policy_interest_rate',
 'JP_Effective_exchange_rate',
 'JP_Equity_prices',
 'JP_Exchange_rate',
 'JP_GDP_Growth_Rate',
 'JP_House_prices_residential',
 'JP_Inflation_rate',
 'JP_Long_term_interest_rate',
 'JP_Oil_price',
 'JP_Unemployment_rate',
 'US_Central_bank_Intervention_rate_policy_interest_rate',
 'US_Effective_exchange_rate',
 'US_Equity_prices',
 'US_GDP_Growth_Rate',
 'US_House_prices_residential',
 'US_Inflation_rate',
 'US_Long_term_interest_rate',
 'US_Oil_price',
 'US_Unemployment_rate']

## STATIONARITÉ

### TESTS DE STATIONNARITE

In [8]:
import numpy as np
import pandas as pd
import warnings

from statsmodels.tsa.stattools import adfuller, kpss

def stationarity_tests_summary(
    df: pd.DataFrame,
    variables=None,
    adf_alpha: float = 0.05,
    kpss_alpha: float = 0.05,
    kpss_regression: str = "c",
    autolag: str = "AIC",
    dropna: bool = True,
    min_n: int = 20
) -> pd.DataFrame:
    """
    Calcule ADF + KPSS pour une liste de variables, renvoie un tableau récapitulatif
    avec p-values, décisions à 5% et diagnostic.

    Parameters
    ----------
    df : DataFrame (wide) avec colonnes = variables.
    variables : list[str] ou None (par défaut: toutes les colonnes numériques).
    adf_alpha : seuil de rejet ADF (H0 = non-stationnaire).
    kpss_alpha : seuil de rejet KPSS (H0 = stationnaire).
    kpss_regression : 'c' (stationnaire autour d'une constante) ou 'ct' (constante+tendance).
    autolag : méthode de sélection du lag pour ADF ('AIC','BIC','t-stat',None).
    dropna : supprime les NA avant tests.
    min_n : taille minimale requise pour lancer les tests.

    Returns
    -------
    DataFrame avec colonnes:
    variable, n, adf_pvalue, kpss_pvalue, ADF_stationary_5pct, KPSS_stationary_5pct, diagnosis
    """
    if variables is None:
        # par défaut: toutes les colonnes numériques
        variables = df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        variables = list(variables)

    results = []

    for var in variables:
        if var not in df.columns:
            results.append({
                "variable": var, "n": 0,
                "adf_pvalue": np.nan, "kpss_pvalue": np.nan,
                "ADF_stationary_5pct": np.nan, "KPSS_stationary_5pct": np.nan,
                "diagnosis": "Missing"
            })
            continue

        s = df[var]
        s = s.dropna() if dropna else s

        # Cast float (évite certains soucis)
        s = pd.to_numeric(s, errors="coerce").dropna()

        n = len(s)

        # Cas série trop courte / constante
        if n < min_n or s.nunique() <= 1:
            results.append({
                "variable": var, "n": n,
                "adf_pvalue": np.nan, "kpss_pvalue": np.nan,
                "ADF_stationary_5pct": np.nan, "KPSS_stationary_5pct": np.nan,
                "diagnosis": "Insufficient/Constant"
            })
            continue

        # -------- ADF --------
        adf_p = np.nan
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                adf_out = adfuller(s.values, autolag=autolag)
            adf_p = float(adf_out[1])
        except Exception:
            adf_p = np.nan

        # -------- KPSS --------
        kpss_p = np.nan
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                kpss_out = kpss(s.values, regression=kpss_regression, nlags="auto")
            kpss_p = float(kpss_out[1])
        except Exception:
            kpss_p = np.nan

        # Décisions
        # ADF: rejeter H0 (non-stationnaire) si p < alpha => stationnaire
        adf_stationary = (adf_p < adf_alpha) if not np.isnan(adf_p) else np.nan
        # KPSS: rejeter H0 (stationnaire) si p < alpha => non stationnaire
        # donc stationnaire si p > alpha
        kpss_stationary = (kpss_p > kpss_alpha) if not np.isnan(kpss_p) else np.nan

        # Diagnostic combiné
        if adf_stationary is True and kpss_stationary is True:
            diag = "Stationary"
        elif adf_stationary is False and kpss_stationary is False:
            diag = "Non-stationary"
        elif (adf_stationary is np.nan) or (kpss_stationary is np.nan):
            diag = "Test failed"
        else:
            diag = "Ambiguous"

        results.append({
            "variable": var,
            "n": n,
            "adf_pvalue": adf_p,
            "kpss_pvalue": kpss_p,
            "ADF_stationary_5pct": adf_stationary,
            "KPSS_stationary_5pct": kpss_stationary,
            "diagnosis": diag
        })

    out = pd.DataFrame(results)

    # Optionnel: trier pour lecture (Non-stationary / Ambiguous / Stationary)
    order = pd.CategoricalDtype(
        categories=["Non-stationary", "Ambiguous", "Stationary", "Insufficient/Constant", "Missing", "Test failed"],
        ordered=True
    )
    if "diagnosis" in out.columns:
        out["diagnosis"] = out["diagnosis"].astype(order)

    return out


In [9]:
variables = df_ts.columns.tolist()
stationarity_df = stationarity_tests_summary(df_ts, variables)
stationarity_df


,variable,n,adf_pvalue,kpss_pvalue,ADF_stationary_5pct,KPSS_stationary_5pct,diagnosis
0,EU_Central_bank_Intervention_rate_policy_inter...,56,6.038112e-01,0.100000,False,True,Ambiguous
1,EU_Effective_exchange_rate,56,3.741599e-01,0.010000,False,False,Non-stationary
2,EU_GDP_Growth_Rate,56,1.582775e-16,0.100000,True,True,Stationary
3,EU_Long_term_interest_rate,56,4.560824e-01,0.022658,False,False,Non-stationary
4,EU_Unemployment_rate,56,8.798221e-01,0.010000,False,False,Non-stationary
5,JP_Central_bank_Intervention_rate_policy_inter...,56,5.561534e-01,0.010000,False,False,Non-stationary
6,JP_Effective_exchange_rate,56,7.671145e-01,0.010000,False,False,Non-stationary
7,JP_Equity_prices,56,9.375750e-01,0.010000,False,False,Non-stationary
8,JP_Exchange_rate,56,9.252683e-01,0.010000,False,False,Non-stationary
9,JP_GDP_Growth_Rate,56,6.454408e-16,0.100000,True,True,Stationary


In [14]:
import re
import numpy as np
import pandas as pd
from statsmodels.tsa.filters.hp_filter import hpfilter

# --- tes helpers ---
def hp_gap(series, lamb=1600):
    s = series.astype(float)
    cycle, trend = hpfilter(s.dropna(), lamb=lamb)
    return cycle.reindex(series.index)

def safe_log(series):
    s = series.astype(float)
    return np.log(s.where(s > 0))

# --- moteur ---
def transform_by_variable_rules(
    df_ts: pd.DataFrame,
    rules: dict,
    hp_lambda: int = 1600,
    region_pattern: str = r"^[A-Z]{2,3}_",   # US_, JP_, EU_ ...
    overwrite: bool = True,
) -> pd.DataFrame:
    """
    rules: dict { "VariableName": {log:bool, hp:bool, diff:bool, hp_suffix:str, out_var:str, ...}, ...}
      - VariableName = partie après le préfixe région (ex: "Unemployment_rate")
      - hp_suffix: par défaut "_hp_gap" (tu peux mettre "_gap" si tu veux)
      - out_var: si tu veux renommer la variable "cible" (rare, optionnel)

    Exemple colonne: "US_Unemployment_rate"
      -> var = "Unemployment_rate"
      -> applique rules["Unemployment_rate"] à chaque région trouvée
    """
    df = df_ts.copy()

    # 1) détecte toutes les colonnes du type REGION_var
    cols = [c for c in df.columns if re.match(region_pattern, c)]
    if not cols:
        return df

    # 2) parse region + variable
    parsed = []
    for c in cols:
        region, var = c.split("_", 1)
        parsed.append((region, var, c))

    regions = sorted(set(r for r, _, _ in parsed))
    vars_found = sorted(set(v for _, v, _ in parsed))

    # 3) applique la règle par variable, pour chaque région où la colonne existe
    for region in regions:
        for var in vars_found:
            base_col = f"{region}_{var}"
            if base_col not in df.columns:
                continue
            if var not in rules:
                continue

            spec = rules[var]
            do_log  = bool(spec.get("log", False))
            do_hp   = bool(spec.get("hp", False))
            do_diff = bool(spec.get("diff", False))
            lamb    = spec.get("hp_lambda", hp_lambda)
            hp_suffix = spec.get("hp_suffix", "_hp_gap")

            # si tu veux renommer la "variable" dans les sorties (optionnel)
            out_var = spec.get("out_var", var)

            current_col = base_col

            # LOG
            if do_log:
                log_col = f"{region}_{out_var}_log"
                if overwrite or log_col not in df.columns:
                    df[log_col] = safe_log(df[base_col])
                current_col = log_col

            # HP GAP
            if do_hp:
                hp_col = f"{current_col}{hp_suffix}"  # conserve l'info log -> ..._log_hp_gap
                if overwrite or hp_col not in df.columns:
                    df[hp_col] = hp_gap(df[current_col], lamb=lamb)
                current_col = hp_col

            # DIFF
            if do_diff:
                diff_col = f"{current_col}_diff"
                if overwrite or diff_col not in df.columns:
                    df[diff_col] = df[current_col].diff()
                current_col = diff_col

    return df

In [15]:
rules = {
    "Unemployment_rate": {"hp": True, "diff": True, "hp_suffix": "_hp_gap"},
    "Long_term_interest_rate": {"hp": True, "diff": True, "hp_suffix": "_hp_gap"},
    "House_prices_residential": {"hp": True, "diff": True, "hp_suffix": "_hp_gap"},
    "Effective_exchange_rate": {"hp": True, "diff": True, "hp_suffix": "_hp_gap"},

    "Central_bank_Intervention_rate_policy_interest_rate": {"diff": True},

    "Equity_prices": {"log": True, "hp": True, "hp_suffix": "_hp_gap"},
    "Oil_price": {"log": True, "hp": True, "hp_suffix": "_hp_gap"},

    # si ta source s'appelle "GDP_Growth_Rate" mais tu veux sortir "GDP_log", etc.
    "GDP_Growth_Rate": {"log": True, "hp": True, "diff": True, "out_var": "GDP", "hp_suffix": "_hp_gap"},
}
df_ts1 = transform_by_variable_rules(df_ts, rules, hp_lambda=1600)

c:\Users\werid\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\filters\hp_filter.py:100: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  trend = spsolve(I+lamb*K.T.dot(K), x, use_umfpack=use_umfpack)
c:\Users\werid\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\filters\hp_filter.py:100: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  trend = spsolve(I+lamb*K.T.dot(K), x, use_umfpack=use_umfpack)
c:\Users\werid\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\filters\hp_filter.py:100: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  trend = spsolve(I+lamb*K.T.dot(K), x, use_umfpack=use_umfpack)
c:\Users\werid\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\filters\hp_filter.py:100: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  trend = spsolve(I+lamb*K.T.dot(K), x, use_umfpack=use_u

In [6]:
from statsmodels.tsa.filters.hp_filter import hpfilter

def hp_gap(series, lamb=1600):
    """
    HP gap (cycle) pour données trimestrielles.
    Retourne une série alignée sur l'index original.
    """
    s = series.astype(float)
    cycle, trend = hpfilter(s.dropna(), lamb=lamb)
    return cycle.reindex(series.index)

In [17]:
df_ts1.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['EU_Central_bank_Intervention_rate_policy_interest_rate',
       'EU_Effective_exchange_rate', 'EU_GDP_Growth_Rate',
       'EU_Long_term_interest_rate', 'EU_Unemployment_rate',
       'JP_Central_bank_Intervention_rate_policy_interest_rate',
       'JP_Effective_exchange_rate', 'JP_Equity_prices', 'JP_Exchange_rate',
       'JP_GDP_Growth_Rate', 'JP_House_prices_residential',
       'JP_Inflation_rate', 'JP_Long_term_interest_rate', 'JP_Oil_price',
       'JP_Unemployment_rate',
       'US_Central_bank_Intervention_rate_policy_interest_rate',
       'US_Effective_exchange_rate', 'US_Equity_prices', 'US_GDP_Growth_Rate',
       'US_House_prices_residential', 'US_Inflation_rate',
       'US_Long_term_interest_rate', 'US_Oil_price', 'US_Unemployment_rate',
       'EU_Central_bank_Intervention_rate_policy_interest_rate_diff',
       'EU_Effective_exchange_rate_hp_gap',
       'EU_Effective_exchange_rate_hp_gap_diff', 'EU_GDP_log',
       'EU_GD

In [16]:
variables = df_ts1.columns.tolist()
stationarity_df = stationarity_tests_summary(df_ts1, variables)
stationarity_df

,variable,n,adf_pvalue,kpss_pvalue,ADF_stationary_5pct,KPSS_stationary_5pct,diagnosis
0,EU_Central_bank_Intervention_rate_policy_inter...,56,6.038112e-01,0.100000,False,True,Ambiguous
1,EU_Effective_exchange_rate,56,3.741599e-01,0.010000,False,False,Non-stationary
2,EU_GDP_Growth_Rate,56,1.582775e-16,0.100000,True,True,Stationary
3,EU_Long_term_interest_rate,56,4.560824e-01,0.022658,False,False,Non-stationary
4,EU_Unemployment_rate,56,8.798221e-01,0.010000,False,False,Non-stationary
...,...,...,...,...,...,...,...
61,US_Long_term_interest_rate_hp_gap_diff,55,9.214832e-07,0.100000,True,True,Stationary
62,US_Oil_price_log,56,5.025619e-01,0.100000,False,True,Ambiguous
63,US_Oil_price_log_hp_gap,56,1.073983e-01,0.100000,False,True,Ambiguous
64,US_Unemployment_rate_hp_gap,56,2.596027e-04,0.100000,True,True,Stationary


In [7]:
df_ts["US_Unemployment_rate_hp_gap"] = hp_gap(df_ts["US_Unemployment_rate"], lamb=1600)
df_ts["US_Unemployment_rate_hp_gap_diff"] = df_ts["US_Unemployment_rate_hp_gap"].diff()
df_ts["US_Long_term_interest_rate_gap"] = hp_gap(df_ts["US_Long_term_interest_rate"], lamb=1600)
df_ts["US_Long_term_interest_rate_gap_diff"] = df_ts["US_Long_term_interest_rate_gap"].diff()
df_ts["US_House_prices_residential_hp_gap"] = hp_gap(df_ts["US_House_prices_residential"], lamb=1600)
df_ts["US_House_prices_residential_hp_gap_diff"] = df_ts["US_House_prices_residential_hp_gap"].diff()
df_ts["US_Effective_exchange_rate_hp_gap"] = hp_gap(df_ts["US_Effective_exchange_rate"], lamb=1600)
df_ts["US_Effective_exchange_rate_hp_gap_diff"] = df_ts["US_Effective_exchange_rate_hp_gap"].diff()
df_ts["US_Central_bank_Intervention_rate_policy_interest_rate_diff"] = df_ts["US_Central_bank_Intervention_rate_policy_interest_rate"].diff()



import numpy as np

def safe_log(series):
    s = series.astype(float)
    return np.log(s.where(s > 0))
df_ts["US_Equity_prices_log"] = safe_log(df_ts["US_Equity_prices"])
df_ts["US_Equity_prices_log_hp_gap"] = hp_gap(df_ts["US_Equity_prices_log"], lamb=1600)

df_ts["US_Oil_price_log"] = safe_log(df_ts["US_Oil_price"])
df_ts["US_Oil_price_log_hp_gap"] = hp_gap(df_ts["US_Oil_price_log"], lamb=1600)
df_ts["US_GDP_log"]= safe_log(df_ts["US_GDP_Growth_Rate"])
df_ts["US_GDP_log_hp_gap"] = hp_gap(df_ts["US_GDP_log"], lamb=1600)
df_ts["US_GDP_log_hp_gap_diff"] = df_ts["US_GDP_log_hp_gap"].diff()

variables = df_ts.columns.tolist()
stationarity_df = stationarity_tests_summary(df_ts, variables)
stationarity_df


stationary_cols = stationarity_df.loc[
    (stationarity_df["diagnosis"]) != "Non-stationary",
    "variable"
].tolist()

df_ts1=df_ts[stationary_cols]


c:\Users\werid\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\filters\hp_filter.py:100: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  trend = spsolve(I+lamb*K.T.dot(K), x, use_umfpack=use_umfpack)
c:\Users\werid\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\filters\hp_filter.py:100: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  trend = spsolve(I+lamb*K.T.dot(K), x, use_umfpack=use_umfpack)
c:\Users\werid\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\filters\hp_filter.py:100: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  trend = spsolve(I+lamb*K.T.dot(K), x, use_umfpack=use_umfpack)
c:\Users\werid\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\filters\hp_filter.py:100: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  trend = spsolve(I+lamb*K.T.dot(K), x, use_umfpack=use_u

In [18]:
stationary_cols = stationarity_df.loc[
    (stationarity_df["diagnosis"]) != "Non-stationary",
    "variable"
].tolist()

df_ts2=df_ts1[stationary_cols]

In [22]:
df_macro_stationary = df_ts2.copy()
df_macro_stationary

var_name,EU_Central_bank_Intervention_rate_policy_interest_rate,EU_GDP_Growth_Rate,JP_GDP_Growth_Rate,JP_Inflation_rate,JP_Oil_price,US_GDP_Growth_Rate,US_Long_term_interest_rate,US_Oil_price,EU_Central_bank_Intervention_rate_policy_interest_rate_diff,EU_Effective_exchange_rate_hp_gap,...,US_GDP_log_hp_gap,US_GDP_log_hp_gap_diff,US_House_prices_residential_hp_gap,US_House_prices_residential_hp_gap_diff,US_Long_term_interest_rate_hp_gap,US_Long_term_interest_rate_hp_gap_diff,US_Oil_price_log,US_Oil_price_log_hp_gap,US_Unemployment_rate_hp_gap,US_Unemployment_rate_hp_gap_diff
date,,,,,,,,,,,,,,,,,,,,,
2010-01-01,1.000000,0.357380,1.242027,-1.166667,77.810000,0.484501,3.716667,76.674837,NaN,3.819471,...,-0.240971,NaN,6.075059,NaN,0.604782,NaN,4.339574,-0.262272,-0.143961,NaN
2010-04-01,1.000000,0.999191,1.377096,-0.933333,80.940000,0.967586,3.490000,78.845036,0.000000,-1.544360,...,0.465834,0.706805,5.401879,-0.673180,0.457614,-0.147168,4.367484,-0.237845,-0.129613,0.014349
2010-07-01,1.000000,0.478303,1.723737,-0.800000,75.440000,0.771085,2.786667,76.675000,0.000000,-1.790578,...,0.254099,-0.211735,3.589721,-1.812158,-0.166599,-0.624213,4.339576,-0.269074,-0.081841,0.047772
2010-10-01,1.000000,0.612048,-0.858797,0.100000,83.370000,0.525110,2.863333,87.033160,0.000000,0.232728,...,-0.114811,-0.368909,2.009748,-1.579973,-0.011853,0.154746,4.466289,-0.145205,0.166192,0.248033
2011-01-01,1.000000,0.961402,-1.433205,-0.533333,98.520000,-0.237205,3.460000,105.369424,0.000000,-0.114745,...,NaN,NaN,-0.827594,-2.837341,0.661290,0.673144,4.657472,0.044092,-0.085291,-0.251483
2011-04-01,1.250000,0.127696,-0.640597,-0.400000,115.960000,0.676582,3.210000,117.541905,0.250000,2.393222,...,0.153627,NaN,-2.560086,-1.732493,0.485613,-0.175677,4.766795,0.153057,0.163826,0.249117
2011-07-01,1.500000,0.086265,2.393091,0.133333,112.220000,-0.022313,2.426667,113.266948,0.250000,1.512919,...,NaN,NaN,-2.880002,-0.319915,-0.226518,-0.712131,4.729747,0.117724,0.313716,0.149890
2011-10-01,1.250000,-0.328476,0.034774,-0.300000,111.940000,1.123049,2.046667,109.978629,-0.250000,0.852474,...,0.674850,NaN,-3.224490,-0.344489,-0.539705,-0.313187,4.700286,0.092499,0.164448,-0.149268
2012-01-01,1.000000,-0.162698,1.521606,0.300000,118.180000,0.838591,2.036667,118.427965,-0.250000,-1.857599,...,0.396404,-0.278446,-3.706151,-0.481661,-0.488408,0.051296,4.774305,0.173649,0.015895,-0.148553


In [33]:
len(df_macro_stationary.columns.tolist())

48

In [27]:
df_macro_stationary.to_csv("data/macro_data/df_macro_stationary.csv", index=True)

In [12]:
stationarity_df


,variable,n,adf_pvalue,kpss_pvalue,ADF_stationary_5pct,KPSS_stationary_5pct,diagnosis
0,US_Central_bank_Intervention_rate_policy_inter...,61,9.969745e-01,0.010000,False,False,Non-stationary
1,US_Effective_exchange_rate,61,9.442045e-01,0.010000,False,False,Non-stationary
2,US_Equity_prices,61,9.988414e-01,0.010000,False,False,Non-stationary
3,US_GDP_Growth_Rate,61,7.219137e-11,0.100000,True,True,Stationary
4,US_House_prices_residential,61,9.306144e-01,0.010000,False,False,Non-stationary
5,US_Inflation_rate,61,5.391757e-01,0.049518,False,False,Non-stationary
6,US_Long_term_interest_rate,61,5.324999e-01,0.100000,False,True,Ambiguous
7,US_Oil_price,61,2.953302e-01,0.100000,False,True,Ambiguous
8,US_Unemployment_rate,61,6.014751e-02,0.010000,False,False,Non-stationary
9,US_Unemployment_rate_hp_gap,61,1.375126e-04,0.100000,True,True,Stationary


In [15]:
len(df_ts1.columns)

18